# Tutorial 21: Publishing to the Hugging Face Hub

`HuggingFacePool` turns the HF Hub into a LAILA pool. Memorize a model manifest into it, and the artifacts appear as a real HF repo you can browse, share, or load with `huggingface_hub` clients.

You will:

- Configure a `HuggingFacePool` with an HF token
- Build a small PyTorch model and a manifest of its weights
- Memorize the manifest into the Hub
- Recover the model from a fresh kernel using just the nickname

**Prerequisites:** `pip install "laila-core[huggingface,torch]"` and an HF token with **write** access to a repo you control. Put it in `secrets.toml`.

In [ ]:
import os
import laila

if os.path.exists("./secrets.toml"):
    laila.read_args("./secrets.toml")
    print("loaded ./secrets.toml")
else:
    print("no secrets.toml found - set HF_TOKEN and HF_REPO_ID before running the next cells")

## Build the pool

`repo_id` is `<user>/<repo>`. `repo_type` defaults to `"model"`; pass `"dataset"` if you're storing datasets. The repo is auto-created on first write.

In [ ]:
from laila.pool import HuggingFacePool

hf = HuggingFacePool(
    repo_id=laila.args.HF_REPO_ID,
    token=laila.args.HF_TOKEN,
    repo_type="model",
    path_prefix="laila_pool",
    nickname="hf",
)
laila.memory.extend(hf, pool_nickname="hf")
print("pool registered, repo:", hf.repo_id)

## A small model to publish

A two-layer CNN keeps the demo small. We capture every parameter as its own entry, then tie them together with a manifest.

In [ ]:
import torch
import torch.nn as nn

from laila.policy.central.memory.schema.manifest import Manifest

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, kernel_size=3)
        self.fc = nn.Linear(8 * 6 * 6, 10)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))

model = TinyCNN()
weights = {name: laila.constant(data=p.detach().clone(), nickname=f"tinycnn.{name}")
           for name, p in model.named_parameters()}
print("captured params:", list(weights.keys()))

## Memorize the manifest

In [ ]:
manifest = Manifest(data=weights, nickname="tinycnn_manifest")
laila.memorize(list(weights.values()), pool_nickname="hf").wait()
manifest.memorize(pool_nickname="hf").wait()

print("manifest gid:", manifest.global_id)
print("browse:       https://huggingface.co/" + hf.repo_id)

## Restore from scratch

After clearing local state, the manifest nickname is enough to rebuild the model. `manifest.realized` fetches every leaf in one batch.

In [ ]:
import laila as _laila
recovered_manifest = laila.remember(
    nickname="tinycnn_manifest", pool_nickname="hf", persist=False,
).wait()
if isinstance(recovered_manifest, list):
    recovered_manifest = recovered_manifest[0]

restored_params = recovered_manifest.realized
print("restored param names:", list(restored_params.keys())[:3], "...")

In [ ]:
new_model = TinyCNN()
state_dict = {name: entry.data for name, entry in restored_params.items()}
new_model.load_state_dict(state_dict)

x = torch.randn(1, 3, 8, 8)
with torch.no_grad():
    print("forward pass output shape:", new_model(x).shape)

## Datasets, too

Set `repo_type="dataset"` and `path_prefix=""` to publish a dataset manifest the same way. The `path_prefix` field separates LAILA-managed files from anything you upload through the regular `huggingface_hub` API in the same repo.

## Summary

- A single `HuggingFacePool` turns an HF Hub repo into a fully addressable LAILA pool.
- Memorize a manifest, share the repo URL, and any consumer with the nickname can rebuild your artifacts.
- `repo_type` and `path_prefix` let one repo host both LAILA artefacts and conventional HF content.